# Image Classification Performance Comparison — Practice Notebook

This notebook is mostly complete. Run every cell in order. One segment
contains a deliberate sequencing bug — your job is to find it, fix it,
and interpret the benchmark results across all four configurations.

In [ ]:
import time
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 1 — Review the Dataset and DataLoader configuration

In [ ]:
class ImageDataset(Dataset):
    """Synthetic image dataset with on-the-fly augmentation."""
    def __init__(self, num_samples=2000, image_size=64, num_classes=10):
        self.num_samples = num_samples
        self.image_size = image_size
        self.num_classes = num_classes
        self.augment = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomCrop(image_size, padding=4),
        ])

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        raw_image = torch.rand(3, self.image_size, self.image_size)
        image = self.augment(raw_image)
        label = torch.randint(0, self.num_classes, (1,)).item()
        return image, label

In [ ]:
dataset = ImageDataset(num_samples=2000, image_size=64, num_classes=10)

# TODO (Step 1): review this configuration against best practice.
# Is there a misconfiguration here that would bottleneck data loading?
# Fix it if so.
dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

print(f"Dataset size: {len(dataset)} | Batches per epoch: {len(dataloader)}")

## Model definition and benchmarking harness (provided — no changes needed)

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

loss_fn = nn.CrossEntropyLoss()

In [ ]:
results = {}

def benchmark(train_step_fn, dataloader, steps=20, label="config"):
    """Times `steps` batches through train_step_fn and records images/sec."""
    data_iter = iter(dataloader)
    images, labels = next(data_iter)
    images, labels = images.to(device), labels.to(device)
    train_step_fn(images, labels)

    total_images = 0
    start = time.time()
    for _ in range(steps):
        try:
            images, labels = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            images, labels = next(data_iter)
        images, labels = images.to(device), labels.to(device)
        train_step_fn(images, labels)
        total_images += images.size(0)
    elapsed = time.time() - start

    images_per_sec = total_images / elapsed
    results[label] = images_per_sec
    print(f"{label}: {images_per_sec:.1f} images/sec")
    return images_per_sec

## Step 2 — Baseline (FP32, no optimizations)

In [ ]:
model_baseline = SimpleClassifier().to(device)
optimizer_baseline = torch.optim.AdamW(model_baseline.parameters(), lr=1e-3)

def baseline_step(images, labels):
    optimizer_baseline.zero_grad(set_to_none=True)
    outputs = model_baseline(images)
    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer_baseline.step()

benchmark(baseline_step, dataloader, steps=20, label="baseline")

## Step 3 — Mixed precision only (correct, provided)

In [ ]:
model_amp = SimpleClassifier().to(device)
optimizer_amp = torch.optim.AdamW(model_amp.parameters(), lr=1e-3)
scaler = GradScaler(device)

def amp_only_step(images, labels):
    optimizer_amp.zero_grad(set_to_none=True)
    with autocast(device_type=device, dtype=torch.float16):
        outputs = model_amp(images)
        loss = loss_fn(outputs, labels)
    scaler.scale(loss).backward()
    scaler.step(optimizer_amp)
    scaler.update()

benchmark(amp_only_step, dataloader, steps=20, label="mixed_precision_only")

## Step 4 — Mixed precision + gradient clipping

Run this cell. Something in the sequence below is out of order —
find it, fix it, and re-run before moving on.

In [ ]:
model_clip = SimpleClassifier().to(device)
optimizer_clip = torch.optim.AdamW(model_clip.parameters(), lr=1e-3)
scaler_clip = GradScaler(device)
max_grad_norm = 1.0

def amp_clip_step(images, labels):
    optimizer_clip.zero_grad(set_to_none=True)
    with autocast(device_type=device, dtype=torch.float16):
        outputs = model_clip(images)
        loss = loss_fn(outputs, labels)
    scaler_clip.scale(loss).backward()

    # Bug to find: these two lines are in the wrong order.
    torch.nn.utils.clip_grad_norm_(model_clip.parameters(), max_grad_norm)
    scaler_clip.unscale_(optimizer_clip)

    scaler_clip.step(optimizer_clip)
    scaler_clip.update()

benchmark(amp_clip_step, dataloader, steps=20, label="mixed_precision_plus_clipping")

## Step 5 — Add torch.compile

In [ ]:
model_clip = torch.compile(model_clip)

benchmark(amp_clip_step, dataloader, steps=20, label="mixed_precision_plus_clipping_plus_compile")

## Step 6 — Results table and report

Run this cell last. Confirm all four rows are populated, then write
your 250–400 word performance comparison report addressed to your
team lead, referencing your own real numbers below.

In [ ]:
import pandas as pd

expected_rows = ["baseline", "mixed_precision_only", "mixed_precision_plus_clipping",
                  "mixed_precision_plus_clipping_plus_compile"]
missing = [r for r in expected_rows if r not in results]
if missing:
    print(f"Missing results for: {missing} — run the corresponding cell(s) above.")

df = pd.DataFrame(list(results.items()), columns=["configuration", "images_per_sec"])
if "baseline" in results:
    df["pct_improvement_vs_baseline"] = (
        (df["images_per_sec"] - results["baseline"]) / results["baseline"] * 100
    ).round(1)
df